In [1]:
# 1. Install dependencies
!pip install pandas numpy scikit-learn xgboost joblib kaggle --quiet

import os
import pandas as pd
import numpy as np
import joblib
from google.colab import files

# 2. Upload kaggle.json API Token
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# 3. Configure Kaggle credentials
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download and unzip the dataset
!kaggle datasets download -d naserabdullahalam/phishing-email-dataset
!unzip -o phishing-email-dataset.zip

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset
License(s): CC-BY-SA-4.0
100% 77.1M/77.1M [00:00<00:00, 88.2MB/s]

Archive:  phishing-email-dataset.zip
  inflating: CEAS_08.csv             
  inflating: Enron.csv               
  inflating: Ling.csv                
  inflating: Nazario.csv             
  inflating: Nigerian_Fraud.csv      
  inflating: SpamAssasin.csv         
  inflating: phishing_email.csv      


In [2]:
import re

# Load the dataset (adjust filename if unzipped differently)
csv_path = [f for f in os.listdir('.') if f.endswith('.csv')][0]
df = pd.read_csv(csv_path)

print(f"Dataset Loaded. Total rows: {len(df)}")
print(f"Columns found: {list(df.columns)}")

# Fill missing values in text fields
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')

# Combine Subject and Body into a single text representation
df['text'] = df['subject'] + " " + df['body']

# Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|\S+\@\S+', ' ', text)  # Remove raw URLs/emails
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)             # Keep letters only
    text = re.sub(r'\s+', ' ', text).strip()             # Remove excess whitespace
    return text

print("Cleaning text data...")
df['clean_text'] = df['text'].apply(clean_text)

# Ensure target label column is integer (0 = legit, 1 = phishing)
df['label'] = df['label'].astype(int)

# Filter out empty text rows
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f"Cleaned dataset rows: {len(df)}")
print("Class distribution:\n", df['label'].value_counts())

Dataset Loaded. Total rows: 39154
Columns found: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
Cleaning text data...
Cleaned dataset rows: 39142
Class distribution:
 label
1    21830
0    17312
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Split Train/Test sets (80/20 stratified split)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

# Feature Extraction: TF-IDF
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

print("Fitting TF-IDF Vectorizer...")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Classifier: XGBoost
print("Training XGBoost Classifier...")
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_vec, y_train)

# Evaluation
y_pred = model.predict(X_test_vec)
y_proba = model.predict_proba(X_test_vec)[:, 1]

print("\n" + "="*50)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"ROC-AUC Score:  {roc_auc_score(y_test, y_proba):.4f}")
print("="*50)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Phishing (1)']))

Fitting TF-IDF Vectorizer...
Training XGBoost Classifier...

Accuracy Score: 99.11%
ROC-AUC Score:  0.9996

Classification Report:
                 precision    recall  f1-score   support

Legitimate (0)       1.00      0.98      0.99      3463
  Phishing (1)       0.99      1.00      0.99      4366

      accuracy                           0.99      7829
     macro avg       0.99      0.99      0.99      7829
  weighted avg       0.99      0.99      0.99      7829



In [5]:
# Create models directory
os.makedirs('backend/models', exist_ok=True)

# Save serialized artifacts
classifier_path = 'backend/models/classifier.pkl'
vectorizer_path = 'backend/models/vectorizer.pkl'

joblib.dump(model, classifier_path)
joblib.dump(vectorizer, vectorizer_path)

print(f"Saved classifier to {classifier_path}")
print(f"Saved vectorizer to {vectorizer_path}")

# Download directly to your local machine
files.download(classifier_path)
files.download(vectorizer_path)

Saved classifier to backend/models/classifier.pkl
Saved vectorizer to backend/models/vectorizer.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>